Imports

In [8]:
import numpy as np
import pandas as pd
import xarray as xr
import pickle
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.metrics import f1_score, precision_score, recall_score, balanced_accuracy_score, average_precision_score
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn_extensions.models.elm import ELMClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from lightgbm import LGBMClassifier
from imblearn.metrics import geometric_mean_score

Code

In [9]:
predictoras_tmp = xr.open_dataset('data/preprocessed/predictoras.nc')
z500 = predictoras_tmp['z500'].values
rh = predictoras_tmp['rh'].values
predictoras = np.stack([z500, rh], axis=-1).reshape(z500.shape[0], -1)
etiquetas = pd.read_parquet('data/preprocessed/etiquetas.parquet')

In [10]:

pareto_history = pickle.load(open("data/history/2026-09-10--11:07:01.pkl", 'rb'))
gnomes = []

pareto_history[-1].sort(key=lambda solution: (solution[1][0] + solution[1][1])/2, reverse=True)
gnomes.append(pareto_history[-1][0][0])
for i in range(3):
    reverse = True
    if i > 1:
        reverse = False
    pareto_history[-1].sort(key=lambda solution: solution[1][i], reverse=reverse)
    gnomes.append(pareto_history[-1][0][0])
    # I think I may have gone too far with this

print(len(gnomes))

4


In [11]:
# Train: de 1993 a 2011 (19 años)
# Test: de 2012 a 2016 (5 años)
idx = 365*19

predictoras_train = []
predictoras_test = []

etiquetas_train = []
etiquetas_test = []


for gnome in gnomes:
    scaler = StandardScaler()

    predictoras_train.append(scaler.fit_transform(predictoras[:idx, gnome]))
    predictoras_test.append(scaler.transform(predictoras[idx:, gnome]))
    print(predictoras_train[-1].shape, predictoras_test[-1].shape)

    etiquetas_train.append(etiquetas.iloc[:idx, 1:3])
    etiquetas_test.append(etiquetas.iloc[idx:, 1:3])
    print(etiquetas_train[-1].shape, etiquetas_test[-1].shape, "\n")

(6935, 33) (1825, 33)
(6935, 2) (1825, 2) 

(6935, 18) (1825, 18)
(6935, 2) (1825, 2) 

(6935, 30) (1825, 30)
(6935, 2) (1825, 2) 

(6935, 1) (1825, 1)
(6935, 2) (1825, 2) 



In [ ]:
titles = ['Mejor balance', 'Mejor F1_Drought', 'Mejor F1_Heatwave', 'Menor num. variables']

random_state = 96

model_configs = [
    ('LGBM', LGBMClassifier, {'random_state': random_state}),
    ('SVC', SVC, {'probability': True, 'random_state': random_state}),
    ('Decision Tree', DecisionTreeClassifier, {'random_state': random_state}),
    ('Random Forest', RandomForestClassifier, {'random_state': random_state}),
    ('Gaussian NB', GaussianNB, {}),
    ('KNN', KNeighborsClassifier, {}),
    ('AdaBoost', AdaBoostClassifier, {'random_state': random_state}),
    ('MLP', MLPClassifier, {'random_state': random_state, 'max_iter': 1000}),
    ('Gradient Boost', GradientBoostingClassifier, {'random_state': random_state}),
    ('ELM', ELMClassifier, {'hidden_layer_size': 60, 'random_state': random_state}),
    ('LDA', LinearDiscriminantAnalysis, {}),
    ('QDA', QuadraticDiscriminantAnalysis, {})
]

results_dr = []
results_hw = []

for i in range(len(gnomes)):
    for objective in ['hw', 'dr']:
        for model_name, func, params in model_configs:
            model = func(**params)
            model.fit(predictoras_train[i], etiquetas_train[i][objective])

            prediction = model.predict(predictoras_test[i])
            probabilities = model.predict_proba(predictoras_test[i])[:, 1]

            entry = {
                'solucion': titles[i],
                'modelo': model_name,
                'f1': f1_score(etiquetas_test[i][objective], prediction),
                'precision': precision_score(etiquetas_test[i][objective], prediction, zero_division=0),
                'recall': recall_score(etiquetas_test[i][objective], prediction, zero_division=0),
                'balanced_accuracy': balanced_accuracy_score(etiquetas_test[i][objective], prediction),
                'geometric_mean': geometric_mean_score(etiquetas_test[i][objective], prediction),
                'pr_auc': average_precision_score(etiquetas_test[i][objective], probabilities),
            }
            
            if objective == 'dr':
                results_dr.append(entry)
            else:
                results_hw.append(entry)

results_dr_df = pd.DataFrame(results_dr)
results_hw_df = pd.DataFrame(results_hw)

In [13]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
display(results_dr_df)
display(results_hw_df)

,solucion,modelo,f1,precision,recall,balanced_accuracy,geometric_mean,pr_auc
0,Mejor balance,LGBM,0.659649,0.703242,0.621145,0.767174,0.753148,0.740164
1,Mejor balance,SVC,0.674757,0.751351,0.612335,0.772615,0.755807,0.774419
2,Mejor balance,Decision Tree,0.572354,0.561441,0.583700,0.716358,0.703968,0.431275
3,Mejor balance,Random Forest,0.651797,0.745042,0.579295,0.756825,0.735709,0.742742
4,Mejor balance,Gaussian NB,0.628524,0.503989,0.834802,0.781369,0.779540,0.687571
5,Mejor balance,KNN,0.586982,0.634271,0.546256,0.720976,0.699485,0.587750
6,Mejor balance,AdaBoost,0.686891,0.675906,0.698238,0.793685,0.787925,0.715308
7,Mejor balance,MLP,0.645092,0.632135,0.658590,0.765838,0.758291,0.669097
8,Mejor balance,Gradient Boost,0.684642,0.731830,0.643172,0.782563,0.770049,0.756866
9,Mejor balance,ELM,0.661196,0.706767,0.621145,0.767903,0.753749,0.739605


,solucion,modelo,f1,precision,recall,balanced_accuracy,geometric_mean,pr_auc
0,Mejor balance,LGBM,0.321212,0.679487,0.210317,0.597212,0.454945,0.557385
1,Mejor balance,SVC,0.275410,0.792453,0.166667,0.579837,0.406818,0.602027
2,Mejor balance,Decision Tree,0.435955,0.502591,0.384921,0.661945,0.601190,0.278389
3,Mejor balance,Random Forest,0.310127,0.765625,0.194444,0.592454,0.438851,0.609315
4,Mejor balance,Gaussian NB,0.506631,0.380478,0.757937,0.780113,0.779797,0.493235
5,Mejor balance,KNN,0.270270,0.555556,0.178571,0.577843,0.417714,0.370659
6,Mejor balance,AdaBoost,0.334347,0.714286,0.218254,0.602134,0.463898,0.572307
7,Mejor balance,MLP,0.413965,0.557047,0.329365,0.643704,0.561734,0.476719
8,Mejor balance,Gradient Boost,0.403409,0.710000,0.281746,0.631655,0.525882,0.607064
9,Mejor balance,ELM,0.398844,0.734043,0.273810,0.628958,0.519093,0.606904
